# Stock History Loader

This notebook pulls stock history into the local AlphaLens bronze/silver/gold stores.
It uses the notebook-oriented loader in `fg.pipelines.historical_loader`, including the SEC canonicalization pass needed for live `companyfacts` payloads.

In [3]:
from fg.pipelines import HistoricalDataLoader
from fg.settings import get_settings

settings = get_settings()
loader = HistoricalDataLoader(settings)
settings.data_dirs

ValueError: conf/app.yml ui.lookback_options must be [5, 10, 15, 20]

In [ ]:
TICKERS = ["AAPL", "MSFT", "KO"]
WATCHLIST_NAME = None
USE_FIXTURES = False
BUILD_GOLD = True
LOOKBACK_YEARS = 20
PE_METHOD = "static_15"
SHOW_ESTIMATES = True
MANUAL_PE = None
CONTINUE_ON_ERROR = True

In [ ]:
if WATCHLIST_NAME:
    TICKERS = [
        str(ticker).upper()
        for ticker in settings.watchlists_config.get("watchlists", {}).get(WATCHLIST_NAME, [])
    ]

summary_df = loader.load_tickers(
    TICKERS,
    fixture_mode=USE_FIXTURES,
    build_gold=BUILD_GOLD,
    lookback_years=LOOKBACK_YEARS,
    pe_method=PE_METHOD,
    show_estimates=SHOW_ESTIMATES,
    manual_pe=MANUAL_PE,
    continue_on_error=CONTINUE_ON_ERROR,
)
summary_df

In [ ]:
loader.table_inventory()

In [ ]:
created_views = loader.register_duckdb_views()
created_views

In [ ]:
loader.query_duckdb(
    """
    SELECT ticker, company_key, issuer_name, last_sec_pull_at, last_yahoo_pull_at, last_fmp_pull_at
    FROM v_dim_company
    ORDER BY ticker
    """
)

In [ ]:
loader.query_duckdb(
    """
    SELECT company_key, metric_code, COUNT(*) AS row_count
    FROM v_fundamental_annual
    GROUP BY company_key, metric_code
    ORDER BY company_key, metric_code
    """
)